# Ejercicio 6

In [1]:
import numpy as np
from sklearn.neighbors import LocalOutlierFactor
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, OneHotEncoder, StandardScaler, OrdinalEncoder, TargetEncoder, FunctionTransformer
from sklearn.neighbors import LocalOutlierFactor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier


## Inciso 1

In [2]:
%pip install gdown

#carpeta con los datos de Google Drive
import sys
!{sys.executable} -m gdown --folder 1GOJ63mZ6qZGdbb8v5v6bcy3fsbLs8yju

#cargar el dataset en un DataFrame

import pandas as pd
df = pd.read_pickle('data/house_prices.pkl')
df.head()


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Processing file 1wOLyprsRSgN7TD_MRKKQi_4ts4xdXEAG data_description.txt
Processing file 1Ub5YOgvDgOb3WaTeYLs66f3MaPaQ36DF house_prices.pkl


Retrieving folder contents
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1wOLyprsRSgN7TD_MRKKQi_4ts4xdXEAG
To: l:\FACULTAD DE MIERDA\TERCERO\LABORTARIO DE DATOS 2\lab 2 - unidad 2\data\data_description.txt

  0%|          | 0.00/13.4k [00:00<?, ?B/s]
100%|██████████| 13.4k/13.4k [00:00<?, ?B/s]
Downloading...
From: https://drive.google.com/uc?id=1Ub5YOgvDgOb3WaTeYLs66f3MaPaQ36DF
To: l:\FACULTAD DE MIERDA\TERCERO\LABORTARIO DE DATOS 2\lab 2 - unidad 2\data\house_prices.pkl

  0%|          | 0.00/140k [00:00<?, ?B/s]
100%|██████████| 140k/140k [00:00<00:00, 2.04MB/s]
Download completed


,SalePrice,MSZoning,Neighborhood,LotFrontage,LotArea,OverallCond,YearBuilt,FullBath,BedroomAbvGr,GarageQual,GarageArea,PoolArea,PoolQC,Fence
0,87500,RL,NAmes,NaN,8544,4,1949,2,2,TA,400,0,NaN,NaN
1,164990,RL,CollgCr,65.0,8767,5,2005,2,3,TA,400,0,NaN,NaN
2,144000,RM,CollgCr,NaN,4435,5,2003,1,1,TA,420,0,NaN,NaN
3,307000,rL,Somerst,75.0,10084,5,2004,2,3,TA,636,0,NaN,NaN
4,130500,RL,Sawyer,NaN,13517,8,1976,2,3,TA,475,0,NaN,NaN


Todas las operaciones que utilizen medidas "estadisticas" o utilizen datos de otras observaciones del dataset, introducen data leakeage si se hacen con los datos de testeo.

Corregir errores de tipeo/ inconsistencias en nomenclatura, se lo hacemos al conjunto completo, ya que no fuga ningun dato.

El borrado de duplicados deberia de hacerse con el dataset completo: Si el dato que esta duplicado, cae en el conjunto de entrenamiento y en el conjunto de testeo, el modelo podria memorizar la respuesta.

In [3]:
df['SalePrice']=df['SalePrice'].astype(float)

#reemplazamos los "NA" por un nan
df['LotFrontage']=df['LotFrontage'].replace('NA',np.nan)
#por las dudas convertimos todos a numericos
df['LotFrontage']=df['LotFrontage'].astype(float)

print("Tipo final:", df['LotFrontage'].dtype)
print("Nulos reales:", df['LotFrontage'].isna().sum())

Tipo final: float64
Nulos reales: 264


C:\Users\lioju\AppData\Local\Temp\ipykernel_9968\1928462393.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['LotFrontage']=df['LotFrontage'].replace('NA',np.nan)


In [4]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['SalePrice'])
y = df['SalePrice']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Entrenamiento: {X_train.shape[0]} filas.")
print(f"Prueba: {X_test.shape[0]} filas.")


Entrenamiento: 1040 filas.
Prueba: 447 filas.


## Inciso 2

El Paso a Paso Teórico de un Pipeline

- Imputación de Nulos: No puedes escalar ni codificar texto si hay celdas vacías (NaN). Los algoritmos crashean de inmediato.
    - Numéricas: Se rellenan con Mediana, Media o KNN.
    - Categóricas: Se rellenan con la moda (el más frecuente) o con una palabra fija (ej: "None" o "NA").
- Codificación de Texto (Encoders): Los algoritmos solo entienden números. Aquí transformas las categorías imputadas en valores matemáticos.
    - Categóricas: Se aplica One-Hot Encoding, Ordinal Encoding o label encoding.
- Transformaciones de Forma (Opcional): Si tenemos asimetría (como vimos en el Ej 5), aquí se aplica Yeo-Johnson o Logaritmo a las numéricas.
- Escalado de Numéricas: Se aplican el RobustScaler o MinMaxScaler. Tiene que ir al final, porque si lo haces antes de la imputación o de la transformación de forma, arruinas las escalas.

In [5]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler, FunctionTransformer
from category_encoders import TargetEncoder # Librería extra para Target Encoding
from sklearn.preprocessing import PowerTransformer

# DEFINICIÓN DE LISTAS DE COLUMNAS (Clasificación por tipo de dato)
# NUMERICAS
numeric_cols = X_train.select_dtypes(include=['float64', 'int64']).columns.tolist()
#Nominales 
col_zoning = ['MSZoning']
col_fence = ['Fence']
col_neighborhood = ['Neighborhood'] # Alta cardinalidad
#Ordinales
col_garage_qual = ['GarageQual']
col_pool = ['PoolQC']


#CREACIÓN DE SUB-PIPELINES
# Sub-pipeline Numérico
numeric_transformer = Pipeline(steps=[
    ('imputer', KNNImputer(n_neighbors=5)), # Lógica del Ejercicio 2
    ('scaler', RobustScaler()),
    ('power_transformer',PowerTransformer(method='yeo-johnson'))   # Lógica del Ejercicio 5 (ideal para los outliers de LotArea)
])
#El RobustScaler ignora los valores atípicos usando cuartiles (para que los datos normales no colapsen),
#y el Yeo-Johnson "comprime" las colas largas donde viven esos outliers extremos

# Sub-pipelines Nominales de BAJA cardinalidad (Se usa One-Hot Encoding)
pipe_zoning = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

pipe_fence = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='None')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Sub-pipeline Nominal de ALTA cardinalidad (Se usa Target Encoding - Ejercicio 4)
pipe_neighborhood = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('target_enc', TargetEncoder(smoothing=10)) 
])

#  Sub-pipeline Ordinal (Se mapea preservando el orden manual - Ejercicio 4)
garage_order = ['NA', 'PO', 'FA', 'TA', 'GD', 'EX']
pipe_garage = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
    ('ordinal', OrdinalEncoder(categories=[garage_order], handle_unknown='use_encoded_value', unknown_value=-1))
])

#  Sub-pipeline Especial (Se crea una función personalizada para binarizar)
def binarize_sparsity(X):
    return (~pd.isna(X)).astype(int)

pipe_pool = Pipeline(steps=[
    ('binarizer', FunctionTransformer(binarize_sparsity))
])


#EL COLUMN TRANSFORMER
# Junta todos los sub-pipelines y los aplica a las columnas correspondientes
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('zoning', pipe_zoning, col_zoning),
        ('fence', pipe_fence, col_fence),
        ('neighborhood', pipe_neighborhood, col_neighborhood),
        ('garage', pipe_garage, col_garage_qual),
        ('pool', pipe_pool, col_pool)
    ],
    remainder='drop' # Columnas que no especificamos se descartan
)

#VISUALIZACIÓN INTERACTIVA
from sklearn import set_config
set_config(display="diagram")

# Al llamar a la variable sola al final de la celda, Jupyter dibuja el diagrama
preprocessor


,transformers,"[('num', ...), ('zoning', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,n_neighbors,5
,weights,'uniform'


# Inciso 3

In [7]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

# UNIR PREPROCESADOR CON MODELO PREDICTIVO
# La consigna pide un "modelo de regresión simple". Usamos Ridge
# porque penaliza coeficientes muy grandes.
pipeline_final = Pipeline(steps=[
    ('preprocessor', preprocessor), # Todo el pipeline del inciso 2
    ('regresor', Ridge(random_state=42))
])


# DEFINIMOS EL ESPACIO DE BÚSQUEDA
# usamos la sintaxis: 'nombrepaso__nombreparametro'
param_grid = {
    # Prueba A: ¿Es mejor escalar con RobustScaler o con la clásica Estandarización?
    'preprocessor__num__scaler': [RobustScaler(), StandardScaler()],
    
    # Prueba B: ¿Qué imputación numérica funciona mejor? ¿KNN o la Mediana simple?
    'preprocessor__num__imputer': [KNNImputer(n_neighbors=5), SimpleImputer(strategy='median')],
    
    # Prueba C: ¿Aplicar o no aplicar la transformación de Yeo-Johnson?
    # Según la Pág 25 del PDF, podemos pasar [None] para pedirle que "salte" este paso y ver si empeora.
    'preprocessor__num__power_transformer': [PowerTransformer(method='yeo-johnson'), None],
    
    # Prueba D: Ajuste del modelo predictivo
    'regresor__alpha': [0.1, 1.0, 10.0]
}


# CONFIGURAR Y EJECUTAR LA VALIDACIÓN CRUZADA
# GridSearchCV probará TODAS las combinaciones
# con 5 "folds" (cv=5)
grid_search = GridSearchCV(
    estimator=pipeline_final,
    param_grid=param_grid,
    cv=5,      #Divide el entrenamiento en 5 partes 
    scoring='r2',   #Usamos R-cuadrado para medir la calidad
    n_jobs=-1,  # -1 significa: "Usa todos los procesadores de la PC"
    verbose=1   # Muestra una barra de progreso
)


# Se ajusta SOLO sobre los datos de entrenamiento para evitar el Data Leakage
grid_search.fit(X_train, y_train)


#IMPRIMIR RESULTADOS Y EVALUAR EN TEST
print("\n--- RESULTADOS DE LA VALIDACIÓN CRUZADA ---")
print(f"Mejor R-cuadrado en Entrenamiento (Cross-Validation): {grid_search.best_score_:.4f}")
print("Configuración ganadora de preprocesamiento:")
for param, value in grid_search.best_params_.items():
    print(f" - {param}: {value}")

# Finalmente, usar el mejor modelo elegido para predecir el futuro (X_test)
score_test = grid_search.score(X_test, y_test)
print(f"\nR-cuadrado final en conjunto de Prueba: {score_test:.4f}")


Fitting 5 folds for each of 24 candidates, totalling 120 fits

--- RESULTADOS DE LA VALIDACIÓN CRUZADA ---
Mejor R-cuadrado en Entrenamiento (Cross-Validation): 0.6591
Configuración ganadora de preprocesamiento:
 - preprocessor__num__imputer: KNNImputer()
 - preprocessor__num__power_transformer: PowerTransformer()
 - preprocessor__num__scaler: StandardScaler()
 - regresor__alpha: 10.0

R-cuadrado final en conjunto de Prueba: 0.6222


### Conclusion

1. La Imputación Inteligente vale la pena (Ganó KNN)
El modelo eligió el KNNImputer() por encima de la simple Mediana. Esto confirma empíricamente la hipótesis del Ejercicio 2: buscar "casas vecinas" similares para adivinar el valor faltante de LotFrontage retiene muchísima más información útil que simplemente rellenar los huecos con el valor central del barrio.

2. La Asimetría daña a los modelos lineales (Ganó Yeo-Johnson)
La opción ganadora incluyó el PowerTransformer() (en vez de None). Esto demuestra lo que vimos en el Ejercicio 5: las variables como LotArea tenían colas gigantescas (outliers extremos). Los modelos de regresión asumen distribuciones normales (campanas de Gauss). Al aplicar Yeo-Johnson, curamos esa asimetría, ayudando al modelo lineal a capturar la tendencia real en vez de confundirse con valores extremos.

3. La sorpresa del Escalador: El trabajo en equipo (Ganó StandardScaler)
Pensamos que iba a ganar RobustScaler (porque en el Ej 5 dijimos que era mejor para outliers). ¿Por qué ganó StandardScaler? Como el paso anterior (Yeo-Johnson) ya se encargó de aplastar los outliers y normalizar la forma, **la variable ya no tiene valores atípicos severos**. Una vez que la curva está limpia y normalizada, el StandardScaler funciona a la perfección.

4. Mucha dimensionalidad requiere frenos (Ganó Alpha = 10.0)
El modelo eligió el parámetro de regularización más alto que le dimos (alpha=10.0). Como creamos decenas de columnas nuevas al usar el OneHotEncoder (para MSZoning y Fence), el modelo corría riesgo de sobreajustarse (aprenderse de memoria las columnas). El alpha alto actúa como un "freno", penalizando los coeficientes grandes para que el modelo sea más general y estable.

5. No hay Data Leakage
La métrica bajó de 0.6591 (en el CV de Entrenamiento) a 0.6222 (en la Prueba invisible). Esa caída de 0.03 puntos es completamente normal y sana. Si hubiéramos cometido el error de Data Leakage, el puntaje de entrenamiento habría sido altísimo e irreal (ej: 0.90) y el de prueba se habría desplomado (ej: 0.30). Estos números demuestran que nuestro ColumnTransformer encapsuló todo perfectamente y el modelo aprendió reglas reales y generalizables.


# Inciso 4

In [9]:
import pickle
import numpy as np

# 1. EXTRACCIÓN DEL MEJOR PIPELINE
#Extraemos el pipeline ganador 
# (el que ya tiene el mejor escalador, imputador y modelo entrenado con X_train).
mejor_pipeline = grid_search.best_estimator_

# APLICACIÓN SIN DATA LEAKAGE
# Usamos .predict() sobre X_test.
# asegurando que TODAS las imputaciones y escalados usen la matemática del X_train.
predicciones_test = mejor_pipeline.predict(X_test)

print(f"Ejemplo de las primeras 3 predicciones: {np.round(predicciones_test[:3], 2)}")


# Guardamos el pipeline completo (preprocesador + modelo) en un archivo binario.
nombre_archivo = 'pipeline_casas_produccion.pkl'

with open(nombre_archivo, 'wb') as archivo:
    pickle.dump(mejor_pipeline, archivo)
print(f"\nPipeline guardado como: '{nombre_archivo}'")


#SIMULACIÓN(Carga y uso)
# Imaginemos que estamos en otra computadora o servidor y cargamos el archivo...
with open(nombre_archivo, 'rb') as archivo:
    pipeline_cargado = pickle.load(archivo)

print("\n Pipeline cargado.")

# Demostramos que el pipeline cargado es robusto prediciendo la primera casa del test
casa_nueva = X_test.iloc[[0]] # Tomamos 1 sola fila para simular un cliente nuevo
precio_estimado = pipeline_cargado.predict(casa_nueva)

print(f"Predicción para el cliente nuevo usando el archivo cargado: ${precio_estimado[0]:,.2f}")


Ejemplo de las primeras 3 predicciones: [249184.68 225848.36 106104.08]

Pipeline guardado como: 'pipeline_casas_produccion.pkl'

 Pipeline cargado.
Predicción para el cliente nuevo usando el archivo cargado: $249,184.68
